# EuroCrops retraining pipeline

Standalone companion to `crop_classifier_training.ipynb` in this same directory — split out so it can be run independently without any of the older BreizhCrops/wheat-mustard cells. Fill in your CDSE credentials in the Setup cell below before running (see the comment there for safer alternatives to editing this file directly).

The cells above train on BreizhCrops' pre-packaged, frozen Sentinel-2 archive (2017 imagery,
extracted and calibrated once, years ago). The deployed app fetches live Sentinel-2 data through
the Copernicus Data Space Ecosystem (CDSE) Statistical API at prediction time, which serves
scenes under whatever processing baseline is *currently* live — confirmed to be a materially
different radiometric calibration than BreizhCrops' original archive (checked directly via
product IDs, e.g. a 2017 scene reprocessed under baseline N0500 in October 2023). That mismatch
between train-time and inference-time data sources causes wrong predictions in production even
though the model itself is sound (verified: feeding it BreizhCrops' own exact training-format
input predicts correctly).

The fix implemented below: don't train on anyone's frozen archive. Use EuroCrops' labeled parcel
geometries (France, 2018 season) paired with imagery we fetch ourselves, live, through the same
CDSE API the deployed app uses for inference. Training data and inference data are then always
measured by the same instrument, at whatever calibration is current when each happens.

Class scheme: kept identical to the existing 9 classes (barley, wheat, rapeseed, corn,
sunflower, orchards, nuts, permanent meadows, temporary meadows) by mapping directly from the
original French RPG parcel codes EuroCrops preserves alongside its harmonized HCAT labels — the
exact same codes BreizhCrops' own classmapping.csv uses. This was checked directly against
EuroCrops' own France mapping file (csvs/country_mappings/fr_2018.csv in the EuroCrops repo),
not assumed. Note: HCAT's own harmonized taxonomy collapses permanent/temporary meadows into one
category with no split — that's why this uses the original per-country code column instead of
the harmonized HCAT column for the meadow classes specifically.


In [ ]:
# ---- Setup ----
!pip install -q geopandas sentinelhub shapely

import os
import io
import time
import json
import pickle
import zipfile
import random

import numpy as np
import requests
import geopandas as gpd
import torch
import torch.nn as nn
from shapely.geometry import mapping as shapely_mapping
from sentinelhub import CRS, DataCollection, Geometry, SentinelHubStatistical, SHConfig

random.seed(42)
np.random.seed(42)

# Mount Drive so the fetch checkpoint and trained model survive a Colab disconnect -
# the live fetch loop below can realistically run for hours.
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/CropMapping/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- Config ----
# CDSE Sentinel Hub OAuth client — register at the Sentinel Hub Dashboard
# (dataspace.copernicus.eu -> User Settings -> OAuth clients).
#
# Fill these in locally, on your own machine, right before running - do NOT commit
# this file with real values filled in. Two safer options than editing this cell
# directly: (a) use Colab secrets instead (key icon in the left sidebar, then
# `from google.colab import userdata; CDSE_SH_CLIENT_ID = userdata.get(...)`),
# which is what the combined training notebook uses, or (b) if running locally,
# read from environment variables / a gitignored .env file instead of a literal here.
CDSE_SH_CLIENT_ID = "PASTE_YOUR_CLIENT_ID_HERE"
CDSE_SH_CLIENT_SECRET = "PASTE_YOUR_CLIENT_SECRET_HERE"
assert CDSE_SH_CLIENT_ID != "PASTE_YOUR_CLIENT_ID_HERE", "fill in CDSE_SH_CLIENT_ID above before running"
assert CDSE_SH_CLIENT_SECRET != "PASTE_YOUR_CLIENT_SECRET_HERE", "fill in CDSE_SH_CLIENT_SECRET above before running"
CDSE_SH_BASE_URL = "https://sh.dataspace.copernicus.eu"
CDSE_SH_TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

EUROCROPS_YEAR = 2018          # France's currently-available EuroCrops season
EUROCROPS_ZIP_URL = "https://zenodo.org/api/records/8229128/files/FR_2018.zip/content"
EUROCROPS_LOCAL_DIR = "eurocrops_fr_2018"  # local Colab disk is fine for this - re-downloaded each session, not precious

N_PER_CLASS = 300              # stratified sample size per class - tune based on time/PU budget.
                                # Recommended: do a first pass with N_PER_CLASS = 20-30 to confirm
                                # the whole pipeline runs end to end before committing hours to the full run.
TARGET_SEQ_LEN = 45            # must match the deployed model's expectation exactly
MAX_CLOUD_COVER_PERCENT = 60
REFLECTANCE_SCALE = 1e-4
RESOLUTION_METERS = 10

# Exact band order the deployed model expects (verified against breizhcrops' own
# get_default_transform - lexicographic, not natural numeric order). No B2/B10 swap
# workaround here: that swap only existed in BreizhCrops' own stored archive. We're
# fetching real, correctly-labeled bands for both training and inference this time.
SENTINEL2_L1C_BANDS = ["B1", "B10", "B11", "B12", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9"]
_SH_BAND_NAMES = {
    "B1": "B01", "B2": "B02", "B3": "B03", "B4": "B04", "B5": "B05",
    "B6": "B06", "B7": "B07", "B8": "B08", "B8A": "B8A", "B9": "B09",
    "B10": "B10", "B11": "B11", "B12": "B12",
}
_sh_bands = [_SH_BAND_NAMES[b] for b in SENTINEL2_L1C_BANDS]

# Exact same 9-class scheme as the deployed model, mapped from the original French RPG
# codes EuroCrops preserves (verified against EuroCrops' own fr_2018.csv mapping file,
# and identical to backend/breizhcrops_dataset/classmapping.csv already used in this repo).
CODE_TO_CLASS = {
    "ORH": "barley", "ORP": "barley",
    "BTH": "wheat", "BTP": "wheat",
    "CZH": "rapeseed", "CZP": "rapeseed",
    "MID": "corn", "MIE": "corn", "MIS": "corn",
    "TRN": "sunflower",
    "AGR": "orchards", "PFR": "orchards", "PWT": "orchards", "VRG": "orchards",
    "CAB": "nuts", "CTG": "nuts", "NOS": "nuts", "NOX": "nuts", "PIS": "nuts",
    "PPH": "permanent meadows", "PRL": "permanent meadows",
    "PTR": "temporary meadows", "RGA": "temporary meadows",
}
CLASS_NAMES = {
    0: "barley", 1: "wheat", 2: "rapeseed", 3: "corn", 4: "sunflower",
    5: "orchards", 6: "nuts", 7: "permanent meadows", 8: "temporary meadows",
}
CLASS_NAME_TO_ID = {v: k for k, v in CLASS_NAMES.items()}


# ---- Model architecture (self-contained copy - identical to the class defined earlier
# in this notebook, duplicated here so this section can run independently without
# needing the older BreizhCrops/wheat-mustard cells above, which depend on Drive files
# specific to this notebook's original author) ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ImprovedTimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, num_classes, d_model=64, nhead=4, num_layers=3, dim_feedforward=64, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model))
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)


In [ ]:
# ---- Download + load EuroCrops France (2018) ----
# Streamed directly to disk in chunks - never materializes the full ~2.6GB in RAM,
# unlike the earlier version of this cell (which used resp.content and crashed
# Colab sessions by loading the whole file into memory before extraction).

ZIP_LOCAL_PATH = "FR_2018.zip"

def _download_with_resume(url, dest_path, max_retries=12):
    """Stream-download url to dest_path, resuming from a partial file via HTTP
    Range requests and retrying transient connection drops. Empirically not
    optional: a ~2.6GB download over a long-lived connection dropped mid-stream
    in testing, and without this a re-run would silently try to unzip the
    corrupt partial file instead of resuming."""
    for attempt in range(1, max_retries + 1):
        existing_size = os.path.getsize(dest_path) if os.path.exists(dest_path) else 0
        headers = {"Range": f"bytes={existing_size}-"} if existing_size else {}
        try:
            with requests.get(url, stream=True, headers=headers, timeout=60) as resp:
                if existing_size and resp.status_code == 416:
                    # Range Not Satisfiable - the range we asked for starts exactly
                    # at EOF, i.e. the file we already have is already complete
                    return
                if existing_size and resp.status_code == 200:
                    # server ignored our Range request - it doesn\'t support resume,
                    # so start this attempt over from scratch
                    existing_size = 0
                resp.raise_for_status()
                mode = "ab" if existing_size and resp.status_code == 206 else "wb"
                with open(dest_path, mode) as out:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        out.write(chunk)
                content_length = resp.headers.get("Content-Length")

            if content_length is not None:
                expected_size = existing_size + int(content_length) if mode == "ab" else int(content_length)
                actual_size = os.path.getsize(dest_path)
                if actual_size < expected_size:
                    raise IOError(f"download incomplete: got {actual_size} bytes, expected {expected_size}")
            return
        except Exception as e:
            print(f"download attempt {attempt}/{max_retries} failed: {e}")
            if attempt == max_retries:
                raise
            wait = min(60, 2**attempt)
            print(f"retrying in {wait}s...")
            time.sleep(wait)



if not os.path.exists(EUROCROPS_LOCAL_DIR):
    print("Downloading EuroCrops FR_2018.zip (~2.6GB)...")
    _download_with_resume(EUROCROPS_ZIP_URL, ZIP_LOCAL_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_LOCAL_PATH) as zf:
        zf.extractall(EUROCROPS_LOCAL_DIR)
    os.remove(ZIP_LOCAL_PATH)  # free disk space now that it's extracted
    print("Extracted to", EUROCROPS_LOCAL_DIR)

# Find the .shp file inside (EuroCrops ships one shapefile per country/year)
shp_path = None
for root, _, files in os.walk(EUROCROPS_LOCAL_DIR):
    for f in files:
        if f.endswith(".shp"):
            shp_path = os.path.join(root, f)
            break
assert shp_path is not None, "No .shp found - check EUROCROPS_LOCAL_DIR contents"
print("Found shapefile:", shp_path)

# Probe just the schema (1 row - cheap) to confirm the actual column name before doing
# a full read. This avoids loading France's entire multi-million-parcel file into RAM
# just to inspect its columns.
probe = gpd.read_file(shp_path, rows=1)
print("columns:", probe.columns.tolist())

# NOTE: verify this against the columns printed above - EuroCrops preserves the
# original per-country code under a country-specific field name; for France this is
# expected to carry the same RPG codes as CODE_TO_CLASS below (shapefile field names
# are truncated to 10 chars, commonly "CODE_CULTU" for French RPG-derived data,
# matching BreizhCrops' own field name since it's the same source system).
ORIGINAL_CODE_COLUMN = "CODE_CULTU"
assert ORIGINAL_CODE_COLUMN in probe.columns, (
    f"{ORIGINAL_CODE_COLUMN} not found - check the columns above and update this constant"
)


In [ ]:
# ---- Read only the parcels we actually need, map to classes, sample ----
# Reads per RPG code with a hard row cap, not one big query for all 23 codes at once.
# Common crops (wheat, corn, temporary meadows) can have hundreds of thousands of
# parcels nationally - filtering by crop type alone (the previous version of this
# cell) still let those through unbounded and crashed the Colab session on RAM.
# Capping rows per code, one code at a time, bounds peak memory to a small, known
# multiple of N_PER_CLASS regardless of how common any single crop is nationally.
import pandas as pd
import gc

OVERSAMPLE_PER_CODE = max(50, N_PER_CLASS)  # read a bit more than needed per code so sampling has room to work with

frames = []
for code, cls_name in CODE_TO_CLASS.items():
    code_gdf = gpd.read_file(shp_path, where=f"{ORIGINAL_CODE_COLUMN} = '{code}'", rows=OVERSAMPLE_PER_CODE)
    if len(code_gdf) == 0:
        continue
    code_gdf["classname"] = cls_name
    frames.append(code_gdf[["classname", "geometry"]])
    print(f"{code} ({cls_name}): read {len(code_gdf)} parcels (capped at {OVERSAMPLE_PER_CODE} - may not be all that exist nationally)")
    del code_gdf

assert len(frames) > 0, (
    "No parcels matched any known code - check ORIGINAL_CODE_COLUMN and the codes "
    "in CODE_TO_CLASS against the actual data (see the columns printed in the cell above)"
)
gdf_labeled = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)
del frames
gc.collect()

print()
print("total parcels loaded (bounded):", len(gdf_labeled))
print(gdf_labeled["classname"].value_counts())

gdf_labeled = gdf_labeled.to_crs(epsg=4326)

# ---- Stratified sample ----
sampled = (
    gdf_labeled.groupby("classname", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_CLASS, len(g)), random_state=42))
    .reset_index(drop=True)
)
print("sampled parcels:", len(sampled))
print(sampled["classname"].value_counts())

# free the larger intermediate frame now that only the small sample is needed
del gdf_labeled
gc.collect()


In [ ]:
# ---- CDSE Statistical API fetch (mirrors backend/app/services/sentinel_fetch.py) ----

def _sh_config():
    config = SHConfig()
    config.sh_client_id = CDSE_SH_CLIENT_ID
    config.sh_client_secret = CDSE_SH_CLIENT_SECRET
    config.sh_base_url = CDSE_SH_BASE_URL
    config.sh_token_url = CDSE_SH_TOKEN_URL
    return config

def _utm_epsg_for(lon, lat):
    zone = int((lon + 180) / 6) + 1
    return (32600 if lat >= 0 else 32700) + zone

_EVALSCRIPT = f'''
//VERSION=3
function setup() {{
  return {{
    input: [{{
      bands: {json.dumps(_sh_bands + ["dataMask"])},
      units: "DN"
    }}],
    output: [
      {{ id: "bands", bands: {len(SENTINEL2_L1C_BANDS)}, sampleType: "INT16" }},
      {{ id: "dataMask", bands: 1 }}
    ]
  }};
}}
function evaluatePixel(sample) {{
  return {{
    bands: [{", ".join(f"sample.{b}" for b in _sh_bands)}],
    dataMask: [sample.dataMask]
  }};
}}
'''

def fetch_time_series(polygon, start_date, end_date, config, data_collection):
    centroid = polygon.centroid
    geometry = Geometry(shapely_mapping(polygon), crs=CRS.WGS84).transform(
        CRS(_utm_epsg_for(centroid.x, centroid.y))
    )
    request = SentinelHubStatistical(
        aggregation=SentinelHubStatistical.aggregation(
            evalscript=_EVALSCRIPT,
            time_interval=(start_date, end_date),
            aggregation_interval="P1D",
            resolution=(RESOLUTION_METERS, RESOLUTION_METERS),
        ),
        input_data=[SentinelHubStatistical.input_data(data_collection, maxcc=MAX_CLOUD_COVER_PERCENT / 100)],
        geometry=geometry,
        config=config,
    )
    result = request.get_data()[0]

    rows = []
    for interval in result.get("data", []):
        bandsout = interval["outputs"]["bands"]["bands"]
        first = bandsout["B0"]["stats"]
        if first["sampleCount"] == 0 or first["noDataCount"] >= first["sampleCount"]:
            continue
        rows.append([bandsout[f"B{i}"]["stats"]["mean"] * REFLECTANCE_SCALE for i in range(len(SENTINEL2_L1C_BANDS))])
    return np.array(rows, dtype=np.float32)


In [ ]:
# ---- Fetch loop, with checkpointing so a disconnect doesn't lose progress ----
CHECKPOINT_PATH = os.path.join(SAVE_DIR, "eurocrops_fetch_progress.pkl")

sh_config = _sh_config()
data_collection = DataCollection.SENTINEL2_L1C.define_from("EUROCROPS_TRAIN_L1C", service_url=sh_config.sh_base_url)

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        results = pickle.load(f)
    print(f"resuming, {len(results)} parcels already fetched")
else:
    results = {}

from datetime import date
start_date = date(EUROCROPS_YEAR, 1, 1)
end_date = date(EUROCROPS_YEAR, 12, 31)

for i, row in sampled.iterrows():
    key = f"{i}"
    if key in results:
        continue
    try:
        x_raw = fetch_time_series(row.geometry, start_date, end_date, sh_config, data_collection)
        if len(x_raw) == 0:
            continue
        results[key] = {"x_raw": x_raw, "classname": row["classname"]}
    except Exception as e:
        print(f"skip {key}: {e}")
    if i % 50 == 0:
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump(results, f)
        print(f"{i}/{len(sampled)} processed, {len(results)} succeeded")
    time.sleep(0.3)  # stay comfortably under CDSE's 300 req/min cap

with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump(results, f)
print("done:", len(results), "parcels fetched")


In [ ]:
# ---- Preprocess: resample to 45 timesteps + per-sample z-score normalize ----
# Same two-step process verified against the training notebook above and matching
# the deployed backend's inference.py exactly.

def preprocess_sequence(x_raw, target_seq_len=TARGET_SEQ_LEN):
    seq_len = x_raw.shape[0]
    idxs = np.random.choice(seq_len, target_seq_len, replace=seq_len < target_seq_len)
    idxs.sort()
    x_resampled = x_raw[idxs]
    mean = x_resampled.mean(axis=0)
    std = x_resampled.std(axis=0)
    std[std == 0] = 1.0
    return (x_resampled - mean) / std

X_eurocrops, y_eurocrops = [], []
for v in results.values():
    if v["x_raw"].shape[0] < 1:
        continue
    X_eurocrops.append(preprocess_sequence(v["x_raw"]))
    y_eurocrops.append(CLASS_NAME_TO_ID[v["classname"]])

X_eurocrops = np.array(X_eurocrops, dtype=np.float32)
y_eurocrops = np.array(y_eurocrops, dtype=np.int64)
print("X shape:", X_eurocrops.shape, " y shape:", y_eurocrops.shape)


In [ ]:
# ---- Train/test split, DataLoaders ----
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

X_train_ec, X_test_ec, y_train_ec, y_test_ec = train_test_split(
    X_eurocrops, y_eurocrops, test_size=0.2, random_state=42, stratify=y_eurocrops
)

X_train_ec_t = torch.tensor(X_train_ec, dtype=torch.float32)
y_train_ec_t = torch.tensor(y_train_ec, dtype=torch.long)
X_test_ec_t = torch.tensor(X_test_ec, dtype=torch.float32)
y_test_ec_t = torch.tensor(y_test_ec, dtype=torch.long)

batch_size = 128
train_loader_ec = DataLoader(TensorDataset(X_train_ec_t, y_train_ec_t), batch_size=batch_size, shuffle=True)
test_loader_ec = DataLoader(TensorDataset(X_test_ec_t, y_test_ec_t), batch_size=batch_size, shuffle=False)


In [ ]:
# ---- Train ----
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print(
        "\n*** No GPU detected - training will run on CPU and be much slower. ***\n"
        "This is a Colab runtime setting, not something this code can change: go to\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> select a GPU (e.g. T4),\n"
        "then reconnect and re-run from this cell.\n"
    )
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

model_eurocrops = ImprovedTimeSeriesTransformer(
    input_dim=X_eurocrops.shape[2],
    num_classes=len(CLASS_NAMES),
    d_model=64, nhead=4, num_layers=3, dim_feedforward=64, dropout=0.2,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model_eurocrops.parameters(), lr=0.001, weight_decay=0.01)
scheduler = OneCycleLR(optimizer, max_lr=0.005, epochs=20, steps_per_epoch=len(train_loader_ec), pct_start=0.3)

best_acc = 0
SAVE_PATH = os.path.join(SAVE_DIR, "best_transformer_eurocrops.pth")

for epoch in range(20):
    model_eurocrops.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader_ec:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_eurocrops(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_eurocrops.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    model_eurocrops.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader_ec:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model_eurocrops(batch_x)
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    val_acc = correct / total

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_eurocrops.state_dict(), SAVE_PATH)

    print(f"epoch {epoch+1}/20  loss={epoch_loss/len(train_loader_ec):.4f}  val_acc={val_acc:.4f}")

print(f"best val acc: {best_acc:.4f}, saved to {SAVE_PATH}")


## Next steps after this finishes

- `best_transformer_eurocrops.pth` is a drop-in replacement for `best_transformer_breizh.pth`
  in the deployed backend - same architecture, same 9 classes, same input contract.
- IMPORTANT: the deployed `sentinel_fetch.py`'s `_SH_BAND_NAMES` currently cross-wires B2/B10
  on purpose, to match a quirk in BreizhCrops' own stored data. This new model trains on real,
  correctly-labeled bands (no swap) - deploying this checkpoint REQUIRES reverting that
  cross-wire back to a normal 1:1 band mapping first, or every prediction will silently feed
  the new model B2/B10 swapped, the same class of bug this whole retraining effort exists to
  fix. Don't just swap the checkpoint filename and CHECKPOINT_PATH in config.py - check
  sentinel_fetch.py's band mapping too.
- Recommended before swapping in production: run the same batch-validation approach used
  earlier this session (real held-out EuroCrops parcels, live CDSE fetch, compare predicted
  vs. true label) to confirm accuracy actually improved before replacing the deployed checkpoint.